## Meta-Learning Complete Pipeline

### Step 1: Performance Label Collection (Cell 1)
- Download datasets from OpenML  
- Apply 13+ preprocessing methods to each dataset  
- Evaluate each with LightGBM 5-Fold CV  
- Save results to `performance_labels_large.csv`  

### Step 2: Meta-Feature Extraction (Cell 2)
- Read dataset IDs from performance labels  
- Compute 70 meta-features for each dataset (dimensions, missing values, etc.)  
- Save results to `meta_features_large.csv`  

### Step 3: Regression Model Training (Cell 3)
- Merge performance labels and meta-features  
- Calculate `Score Gain = Method Score - Baseline Score`  
- Train one LightGBM regressor per preprocessing method  
- Evaluate using Win Accuracy (accuracy of predicting +/- gain)  
- Save 13 models to `meta_models_regression/`  

### Step 4: Classification Model Training (Cell 4, optional)
- Find the “best method” for each dataset  
- Train classifier to predict best method  
- Note: Poor performance, only 32% accuracy  

### Step 5: Old Version Regression Training (Cell 5–6)
- Use `performance_labels.csv` and `meta_features_61.csv` (old data)  
- Yields better results due to more complete data coverage  


## Collect performance labels

In [ ]:
import openml
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import LabelEncoder
import preprocessing_function as pf
import os
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Configuration
RESULTS_FILE = "performance_labels_large.csv"
MAX_DATASETS = 300  # Target number of datasets

def get_model(problem_type):
    """Return LightGBM model with default parameters."""
    common_params = {
        "n_estimators": 100,
        "verbosity": -1,
        "n_jobs": 4, # Use 4 cores (too many cause overhead for small data)
        "random_state": 42,
        # SPEED OPTIMIZATIONS
        "max_bin": 63,          # Fewer bins = Much faster histogram building (Default 255)
        "num_leaves": 31,       # Limit complexity
        "subsample": 0.8,       # Bagging to speed up training
        "colsample_bytree": 0.8 # Feature subsampling
    }
    if problem_type == 'regression':
        return lgb.LGBMRegressor(objective='regression', **common_params)
    else:
        return lgb.LGBMClassifier(objective='binary' if problem_type == 'binary' else 'multiclass', **common_params)

def evaluate_dataset(df, target_col, problem_type):
    """Evaluate dataset using 5-Fold CV with LightGBM."""
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # Basic encoding for LightGBM to handle object types
    for col in X.select_dtypes(include=['object']).columns:
        X[col] = X[col].astype('category')
        
    if problem_type != 'regression' and y.dtype == 'object':
        le = LabelEncoder()
        y = le.fit_transform(y)

    model = get_model(problem_type)
    
    if problem_type == 'regression':
        cv = KFold(n_splits=5, shuffle=True, random_state=42)
        scoring = 'neg_root_mean_squared_error'
    else:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scoring = 'accuracy'
        
    try:
        scores = cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=1)
        return np.mean(scores)
    except Exception as e:
        return np.nan

def get_large_dataset_list():
    """
    Get datasets from known benchmarking suites to avoid API bans.
    Strategies:
    1. OpenML-CC18 (Suite 99): ~72 high quality datasets.
    2. AutoML Benchmark (Suite 271): ~104 datasets (overlaps with 99, but has more).
    3. Regression Benchmark (Suite 269): ~More regression tasks.
    """
    print("🔍 Fetching datasets from OpenML Suites (Safer approach)...")
    
    suite_ids = [99, 271] # CC18 and AutoML Benchmark
    task_ids = set()
    
    for sid in suite_ids:
        try:
            suite = openml.study.get_suite(sid)
            task_ids.update(suite.tasks)
        except Exception as e:
            print(f"⚠️ Failed to fetch Suite {sid}: {e}")
            
    # Convert tasks to dataset IDs
    # OpenML API is strict, so we fetch task details in batches or one by one if needed, 
    # but here we can just list tasks. Actually, suite.tasks returns task IDs.
    # We need dataset IDs.
    
    # Efficient way: List all tasks and filter by our ID list
    # But that might trigger the same block. 
    # Better: Get tasks individually or use openml.tasks.list_tasks(status='active') with ID filter
    
    print(f"  Found {len(task_ids)} tasks across suites. Resolving dataset IDs...")
    
    # We build a simple dataframe
    # To be safe, we iterate and get task metadata which is cached usually
    dataset_ids = set()
    
    # Limit to first MAX_DATASETS to be safe
    tasks_to_check = list(task_ids)[:MAX_DATASETS]
    
    # This might be slow but safe. 
    # Optimization: Use list_tasks with id list if supported, or just trust the suite structure.
    # Suite objects usually cache their content.
    
    # Let's try to get a list of tasks efficiently
    try:
        # Requesting tasks in bulk is safer than list_datasets with wildcards
        tasks_batch = openml.tasks.list_tasks(task_id=tasks_to_check, output_format='dataframe')
        if 'did' in tasks_batch.columns:
            dataset_ids.update(tasks_batch['did'].tolist())
        elif 'dataset_id' in tasks_batch.columns:
            dataset_ids.update(tasks_batch['dataset_id'].tolist())
            
        # Create DataFrame expected by main loop
        datasets_df = pd.DataFrame({'did': list(dataset_ids), 'name': [f"Dataset_{i}" for i in dataset_ids]})
        
    except Exception as e:
        print(f"⚠️ Batch task fetch failed ({e}). Falling back to simple iteration (slower but safer).")
        # Fallback: Just return a dummy DF with IDs, the main loop fetches actual data
        datasets_df = pd.DataFrame({'did': list(tasks_to_check), 'name': ["Unknown"] * len(tasks_to_check)})

    print(f"  Final List: {len(datasets_df)} datasets ready to process.")
    return datasets_df

def run_performance_collection():
    print(f"🚀 Starting Large-Scale Performance Label Collection (Target: {MAX_DATASETS} datasets)")
    
    if not os.path.exists(RESULTS_FILE):
        pd.DataFrame(columns=[
            'dataset_id', 'dataset_name', 'problem_type', 
            'method', 'score', 'is_baseline'
        ]).to_csv(RESULTS_FILE, index=False)
    else:
        print(f"  Resuming from existing {RESULTS_FILE}...")

    # Get already processed IDs
    try:
        processed_ids = pd.read_csv(RESULTS_FILE)['dataset_id'].unique().tolist()
    except:
        processed_ids = []

    datasets_df = get_large_dataset_list()
    
    for i, (_, row) in enumerate(datasets_df.iterrows()):
        dataset_id = int(row['did'])
        dataset_name = row['name']
        
        if dataset_id in processed_ids:
            print(f"  Skipping {dataset_name} (ID: {dataset_id}) - Already processed.")
            continue
            
        try:
            print(f"\nProcessing Dataset {i+1}/{len(datasets_df)}: {dataset_name} (ID: {dataset_id})...")
            dataset = openml.datasets.get_dataset(dataset_id, download_data=True)
            
            try:
                df, _, _, _ = dataset.get_data(dataset_format="dataframe")
                target_col = dataset.default_target_attribute
            except:
                print(f"⚠️ Failed to load data for {dataset_name}. Skipping.")
                continue
                
            if target_col is None:
                print(f"⚠️ No target column for {dataset_name}. Skipping.")
                continue
                
            # Determine problem type
            if df[target_col].nunique() > 20 and (df[target_col].dtype == 'float' or df[target_col].dtype == 'int'):
                 # Heuristic for regression vs classification
                 problem_type = 'regression'
            elif df[target_col].nunique() == 2:
                problem_type = 'binary'
            else:
                problem_type = 'multiclass'
            
            print(f"  Type: {problem_type} | Shape: {df.shape}")
            
            results_batch = []
            
            # --- 1. Baseline ---
            baseline_score = evaluate_dataset(df.copy(), target_col, problem_type)
            print(f"  Baseline Score: {baseline_score:.4f}")
            results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'Baseline', 'score': baseline_score, 'is_baseline': True})
            
            if np.isnan(baseline_score): continue

            num_cols = df.select_dtypes(include=np.number).columns.tolist()
            if target_col in num_cols: num_cols.remove(target_col)
            cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
            if target_col in cat_cols: cat_cols.remove(target_col)

            # --- 2. Categorical Methods ---
            if cat_cols:
                # OHE (<10 categories)
                low_card_cols = [c for c in cat_cols if df[c].nunique() < 10]
                if low_card_cols:
                    try:
                        df_ohe = pf.one_hot_encode(df.copy(), columns=low_card_cols)
                        score = evaluate_dataset(df_ohe, target_col, problem_type)
                        results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'OneHotEncoder_LowCard', 'score': score, 'is_baseline': False})
                    except: pass

                # Frequency Encoding
                try:
                    df_freq = pf.frequency_encode(df.copy(), columns=cat_cols)
                    score = evaluate_dataset(df_freq, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'FrequencyEncoder', 'score': score, 'is_baseline': False})
                except: pass

                # Label Encoding
                try:
                    df_le = pf.label_encode(df.copy(), columns=cat_cols)
                    score = evaluate_dataset(df_le, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'LabelEncoder', 'score': score, 'is_baseline': False})
                except: pass

                # Target Encoding
                try:
                    df_te = df.copy()
                    if problem_type != 'regression':
                        y_enc = LabelEncoder().fit_transform(df[target_col])
                    else:
                        y_enc = df[target_col]
                    
                    for c in cat_cols:
                        global_mean = y_enc.mean()
                        agg = df_te.groupby(c)[target_col].agg(['count', 'mean'])
                        counts = agg['count']
                        means = agg['mean']
                        smooth = 10
                        smooth_means = (counts * means + smooth * global_mean) / (counts + smooth)
                        df_te[c] = df_te[c].map(smooth_means)
                    
                    score = evaluate_dataset(df_te, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'TargetEncoder', 'score': score, 'is_baseline': False})
                except: pass

                # Rare Label Encoding
                try:
                    df_rare = df.copy()
                    for c in cat_cols:
                        vc = df_rare[c].value_counts(normalize=True)
                        rare = vc[vc < 0.05].index
                        df_rare[c] = df_rare[c].replace(rare, 'Other')
                    score = evaluate_dataset(df_rare, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'RareLabelEncoder', 'score': score, 'is_baseline': False})
                except: pass

            # --- 3. Numerical Methods ---
            if num_cols:
                # Log Transform
                try:
                    df_log = pf.apply_power_transform(df.copy(), column=num_cols, method='yeo-johnson')
                    score = evaluate_dataset(df_log, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'LogTransform_YeoJohnson', 'score': score, 'is_baseline': False})
                except: pass
                
                # StandardScaler
                try:
                    from sklearn.preprocessing import StandardScaler
                    df_scaled = df.copy()
                    df_scaled[num_cols] = StandardScaler().fit_transform(df_scaled[num_cols])
                    score = evaluate_dataset(df_scaled, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'StandardScaler', 'score': score, 'is_baseline': False})
                except: pass

                # Binning
                try:
                    from sklearn.preprocessing import KBinsDiscretizer
                    df_bin = df.copy()
                    est = KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='quantile')
                    df_bin[num_cols] = df_bin[num_cols].fillna(df_bin[num_cols].median())
                    df_bin[num_cols] = est.fit_transform(df_bin[num_cols])
                    score = evaluate_dataset(df_bin, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'Binning_Quantile_10', 'score': score, 'is_baseline': False})
                except: pass

            # --- 4. Missing Values ---
            missing_cols = [c for c in df.columns if df[c].isnull().any() and c != target_col]
            if missing_cols:
                # Impute Median/Mode
                try:
                    df_imp = df.copy()
                    num_miss = [c for c in missing_cols if c in num_cols]
                    cat_miss = [c for c in missing_cols if c in cat_cols]
                    if num_miss: df_imp = pf.impute_median(df_imp, num_miss)
                    if cat_miss: df_imp = pf.impute_mode(df_imp, cat_miss)
                    score = evaluate_dataset(df_imp, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'ImputeMedianMode', 'score': score, 'is_baseline': False})
                except: pass

                # Impute Constant
                try:
                    df_const = pf.impute_constant(df.copy(), columns=missing_cols, fill_value=-999)
                    score = evaluate_dataset(df_const, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'ImputeConstant', 'score': score, 'is_baseline': False})
                except: pass

                # Missing Indicator
                try:
                    df_ind = pf.add_missing_indicator(df.copy(), columns=missing_cols)
                    score = evaluate_dataset(df_ind, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'MissingIndicator', 'score': score, 'is_baseline': False})
                except: pass

            # --- 5. Interactions ---
            # GUARD: Skip expensive interactions if too many columns (>2000)
            # Calculating 7000x7000 correlation matrix is extremely slow.
            if len(num_cols) < 2000 and len(num_cols) >= 2:
                try:
                    # Arithmetic Interactions
                    df_inter = pf.create_features_from_correlation_analysis(
                        df.copy(), 
                        correlation_threshold=0.7, 
                        target_column=target_col, # Fix: Pass target_column for smart selection
                        feature_types=['product', 'ratio', 'sum', 'difference'],
                        max_new_features=0.5 # Dynamic: Add up to 50% more features
                    )
                    score = evaluate_dataset(df_inter, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'ArithmeticInteractions', 'score': score, 'is_baseline': False})
                except: pass

            if len(cat_cols) < 500 and len(cat_cols) >= 2:
                try:
                    # Categorical Interactions
                    df_cat_inter = pf.combine_categorical_features(
                        df.copy(), 
                        columns_to_combine=cat_cols[:2],
                        new_col_name='combined_cat',
                        drop_original=False
                    )
                    score = evaluate_dataset(df_cat_inter, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'CategoricalInteractions', 'score': score, 'is_baseline': False})
                except: pass

            # --- 6. Date Features ---
            date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
            if date_cols:
                try:
                    df_date = df.copy()
                    for dc in date_cols:
                        try:
                            df_date = pf.extract_datetime_features(df_date, dc)
                        except: pass
                    score = evaluate_dataset(df_date, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'DateFeatures', 'score': score, 'is_baseline': False})
                except: pass

            # --- 7. Dimensionality Reduction & Selection ---
            # GUARD: Skip expensive transform if too many columns (>3000)
            if num_cols and len(num_cols) > 2 and len(num_cols) < 3000:
                # PCA
                try:
                    from sklearn.decomposition import PCA
                    from sklearn.preprocessing import StandardScaler
                    df_pca = df.copy()
                    X_num = df_pca[num_cols].fillna(df_pca[num_cols].median())
                    # Scaling is crucial for PCA
                    X_num = StandardScaler().fit_transform(X_num)
                    
                    pca = PCA(n_components=0.95)
                    X_pca = pca.fit_transform(X_num)
                    
                    df_pca_new = pd.DataFrame(X_pca, index=df.index)
                    other_cols = [c for c in df.columns if c not in num_cols]
                    df_pca = pd.concat([df_pca_new, df[other_cols]], axis=1)
                    
                    score = evaluate_dataset(df_pca, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'PCA_95', 'score': score, 'is_baseline': False})
                except: pass
                
                # FastICA
                try:
                    df_ica = pf.apply_fastica(df.copy(), replace_ratio=0.4, random_state=42)
                    score = evaluate_dataset(df_ica, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'FastICA', 'score': score, 'is_baseline': False})
                except: pass

            if num_cols and len(num_cols) > 10:
                # SelectKBest
                try:
                    from sklearn.feature_selection import SelectKBest, f_classif, f_regression
                    df_fs = df.copy()
                    X_fs = df_fs.drop(columns=[target_col])
                    y_fs = df_fs[target_col]
                    
                    X_fs_num = X_fs[num_cols].fillna(X_fs[num_cols].median())
                    
                    if problem_type == 'regression':
                        score_func = f_regression
                    else:
                        score_func = f_classif
                        if y_fs.dtype == 'object':
                            y_fs = LabelEncoder().fit_transform(y_fs)
                            
                    k = int(len(num_cols) * 0.5)
                    selector = SelectKBest(score_func=score_func, k=k)
                    selector.fit(X_fs_num, y_fs)
                    
                    selected_cols = [num_cols[i] for i in selector.get_support(indices=True)]
                    keep_cols = selected_cols + cat_cols + [target_col]
                    df_fs = df_fs[keep_cols]
                    
                    score = evaluate_dataset(df_fs, target_col, problem_type)
                    results_batch.append({'dataset_id': dataset_id, 'dataset_name': dataset_name, 'problem_type': problem_type, 'method': 'SelectKBest_50pct', 'score': score, 'is_baseline': False})
                except: pass

            # Save Batch
            if results_batch:
                pd.DataFrame(results_batch).to_csv(RESULTS_FILE, mode='a', header=False, index=False)
                print(f"  ✅ Recorded {len(results_batch)} results.")
                
        except Exception as e:
            print(f"❌ Error processing dataset {dataset_id}: {e}")
            continue

if __name__ == "__main__":
    run_performance_collection()


/Users/q/Code/venv/Teamproject/European-Masters-Team-Project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Starting Large-Scale Performance Label Collection (Target: 300 datasets)
  Resuming from existing performance_labels_large.csv...
🔍 Fetching datasets from OpenML Suites (Safer approach)...
  Found 141 tasks across suites. Resolving dataset IDs...
  Final List: 112 datasets ready to process.
  Skipping Dataset_3 (ID: 3) - Already processed.
  Skipping Dataset_6 (ID: 6) - Already processed.
  Skipping Dataset_40966 (ID: 40966) - Already processed.
  Skipping Dataset_11 (ID: 11) - Already processed.
  Skipping Dataset_12 (ID: 12) - Already processed.
  Skipping Dataset_14 (ID: 14) - Already processed.
  Skipping Dataset_15 (ID: 15) - Already processed.
  Skipping Dataset_16 (ID: 16) - Already processed.
  Skipping Dataset_40975 (ID: 40975) - Already processed.
  Skipping Dataset_18 (ID: 18) - Already processed.
  Skipping Dataset_40979 (ID: 40979) - Already processed.
  Skipping Dataset_40978 (ID: 40978) - Already processed.
  Skipping Dataset_40981 (ID: 40981) - Already processed.
  Sk

## extract meta feature

In [1]:
import openml
import pandas as pd
import numpy as np
import os
from scipy.stats import skew, kurtosis, entropy
import warnings

warnings.filterwarnings('ignore')

# Configuration
PERFORMANCE_FILE = "performance_labels_large.csv"
META_FEATURES_FILE = "meta_features_large.csv"

def calc_stats(series, prefix):
    """Calculate basic stats for a series."""
    if len(series) == 0:
        return {f'{prefix}_Mean': 0, f'{prefix}_Std': 0, f'{prefix}_Skewness': 0, f'{prefix}_Kurtosis': 0}
    return {
        f'{prefix}_Mean': np.mean(series),
        f'{prefix}_Std': np.std(series),
        f'{prefix}_Skewness': skew(series, nan_policy='omit'),
        f'{prefix}_Kurtosis': kurtosis(series, nan_policy='omit')
    }

def extract_features_for_dataset(dataset_id):
    try:
        dataset = openml.datasets.get_dataset(dataset_id, download_data=True)
        df, _, _, _ = dataset.get_data(dataset_format="dataframe")
        target_col = dataset.default_target_attribute
        
        if target_col is None: return None
        
        features = {'dataset_id': dataset_id, 'dataset_name': dataset.name}
        
        # Basic Dimensions
        n_instances, n_attrs = df.shape
        features['Number_of_Instances'] = n_instances
        features['Number_of_Attributes'] = n_attrs
        
        # Missing Values
        n_missing = df.isnull().sum().sum()
        features['Number_of_Missing_Values'] = n_missing
        features['Percentage_of_Missing_Values'] = n_missing / (n_instances * n_attrs) if n_instances * n_attrs > 0 else 0
        
        # Column Types
        num_cols = df.select_dtypes(include=np.number).columns.tolist()
        cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        
        if target_col in num_cols: num_cols.remove(target_col)
        if target_col in cat_cols: cat_cols.remove(target_col)
            
        features['Number_of_Numeric_Attributes'] = len(num_cols)
        features['Number_of_Categorical_Attributes'] = len(cat_cols)
        features['Number_of_Binary_Attributes'] = 0 
        features['Percentage_of_Categorical_Attributes'] = len(cat_cols) / n_attrs if n_attrs > 0 else 0
        features['Percentage_of_Binary_Attributes'] = 0

        # Numeric Stats
        if num_cols:
            means = df[num_cols].mean()
            stds = df[num_cols].std()
            skews = df[num_cols].apply(lambda x: skew(x, nan_policy='omit'))
            kurts = df[num_cols].apply(lambda x: kurtosis(x, nan_policy='omit'))
            
            features.update(calc_stats(means, 'Means_of_Continuous_Attributes'))
            features.update(calc_stats(stds, 'Std_of_Continuous_Attributes'))
            features.update(calc_stats(skews, 'Skewness_of_Continuous_Attributes'))
            features.update(calc_stats(kurts, 'Kurtosis_of_Continuous_Attributes'))
        else:
            for stat in ['Means', 'Std', 'Skewness', 'Kurtosis']:
                features.update(calc_stats([], f'{stat}_of_Continuous_Attributes'))

        # Categorical Stats
        if cat_cols:
            entropies = []
            distincts = []
            for c in cat_cols:
                vc = df[c].astype(str).value_counts(normalize=True)
                entropies.append(entropy(vc, base=2))
                distincts.append(df[c].nunique())
                
            features.update(calc_stats(entropies, 'Entropy_of_Categorical_Attributes'))
            features.update(calc_stats(distincts, 'Distinct_Values_of_Categorical_Attributes'))
        else:
            for stat in ['Entropy', 'Distinct_Values']:
                features.update(calc_stats([], f'{stat}_of_Categorical_Attributes'))
                
        # Placeholders for complex features
        features['Mean_Mutual_Information'] = 0
        features['Max_Mutual_Information'] = 0
        features['Equivalent_Number_of_Attributes'] = 0
        features['Noise_Signal_Ratio'] = 0
        
        return features
        
    except Exception as e:
        print(f"Error extracting features for {dataset_id}: {e}")
        return None

def run_meta_feature_extraction():
    print("🚀 Starting Meta-Feature Extraction for Large Dataset List...")
    
    # 1. Identify which datasets we need features for
    # We prioritize datasets that we have performance labels for.
    if os.path.exists(PERFORMANCE_FILE):
        print(f"  Reading dataset IDs from {PERFORMANCE_FILE}...")
        df_perf = pd.read_csv(PERFORMANCE_FILE)
        target_ids = df_perf['dataset_id'].unique().tolist()
    else:
        print(f"⚠️ {PERFORMANCE_FILE} not found. Please run 'collect_performance_labels_large.py' first.")
        return

    # 2. Check existing meta-features
    if os.path.exists(META_FEATURES_FILE):
        df_meta = pd.read_csv(META_FEATURES_FILE)
        processed_ids = df_meta['dataset_id'].unique().tolist()
        print(f"  Found existing meta-features for {len(processed_ids)} datasets.")
    else:
        df_meta = pd.DataFrame()
        processed_ids = []

    # 3. Process missing IDs
    ids_to_process = [id for id in target_ids if id not in processed_ids]
    print(f"  Need to process {len(ids_to_process)} new datasets.")
    
    new_features_list = []
    
    for i, ds_id in enumerate(ids_to_process):
        print(f"  Processing {ds_id} ({i+1}/{len(ids_to_process)})...")
        feats = extract_features_for_dataset(ds_id)
        if feats:
            new_features_list.append(feats)
            
        # Save periodically
        if len(new_features_list) >= 10:
            df_new = pd.DataFrame(new_features_list)
            df_new.to_csv(META_FEATURES_FILE, mode='a', header=not os.path.exists(META_FEATURES_FILE), index=False)
            print(f"  ✅ Saved batch of {len(new_features_list)} features.")
            new_features_list = []
            
    # Save remaining
    if new_features_list:
        df_new = pd.DataFrame(new_features_list)
        df_new.to_csv(META_FEATURES_FILE, mode='a', header=not os.path.exists(META_FEATURES_FILE), index=False)
        print(f"  ✅ Saved final batch of {len(new_features_list)} features.")

if __name__ == "__main__":
    run_meta_feature_extraction()


🚀 Starting Meta-Feature Extraction for Large Dataset List...
  Reading dataset IDs from performance_labels_large.csv...
  Found existing meta-features for 110 datasets.
  Need to process 0 new datasets.


## train regression

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
LABELS_FILE = "performance_labels_large.csv"  # The output from collection
META_FEATURES_FILE = "meta_features_large.csv" # You need to run extract_meta_features_large.py first
MODEL_DIR = "meta_models_regression"

os.makedirs(MODEL_DIR, exist_ok=True)

def load_and_prepare_data():
    """
    Load labels and meta-features, merge them, and calculate Score Gain.
    """
    print("📂 Loading data...")
    if not os.path.exists(LABELS_FILE) or not os.path.exists(META_FEATURES_FILE):
        print("❌ Data files not found. Please run collection and extraction scripts first.")
        return None

    # Load Performance Labels
    df_lab = pd.read_csv(LABELS_FILE)
    
    # Filter out failures
    df_lab = df_lab.dropna(subset=['score'])
    
    # Pivot to get Baseline Score per dataset
    # We explicitly look for 'Baseline' method or is_baseline=True
    baseline_scores = df_lab[df_lab['method'] == 'Baseline'].set_index('dataset_id')['score']
    
    if baseline_scores.empty:
        print("⚠️ No Baseline scores found. Cannot calculate gains.")
        return None

    # Calculate Gain
    # Gain = Method Score - Baseline Score
    # For Regression (RMSE), Lower is Better? 
    # The collection script handles scoring:
    # Classification: Accuracy (Higher is better)
    # Regression: Neg RMSE (Higher is better, e.g., -0.2 > -0.5)
    # So strictly: Gain = Method - Baseline > 0 means improvement.
    
    df_lab['baseline_score'] = df_lab['dataset_id'].map(baseline_scores)
    df_lab['gain'] = df_lab['score'] - df_lab['baseline_score']
    
    # Drop rows where baseline itself is missing for some reason
    df_lab = df_lab.dropna(subset=['gain'])
    
    # Load Meta-Features
    df_meta = pd.read_csv(META_FEATURES_FILE)
    
    # Merge
    # We want one row per (Dataset, Method)
    full_df = pd.merge(df_lab, df_meta, on='dataset_id', how='inner')
    
    print(f"✅ Loaded {len(full_df)} samples ({full_df['dataset_id'].nunique()} datasets).")
    return full_df

def train_method_regressors(df):
    """
    Train one Regressor per Preprocessing Method.
    Target: 'gain'
    Features: Meta-features
    """
    # Identify Meta-Features (drop metadata cols and non-numeric cols)
    metadata_cols = ['dataset_id', 'dataset_name', 'dataset_name_x', 'dataset_name_y', 'problem_type', 'method', 'score', 'is_baseline', 'baseline_score', 'gain']
    feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in metadata_cols]
    
    print(f"🧠 Meta-Features ({len(feature_cols)}): {feature_cols[:5]}...")
    
    methods = df['method'].unique()
    methods = [m for m in methods if m != 'Baseline'] # Don't predict baseline gain (it's 0)
    
    results = []
    
    print("\n🏋️ Training Regressors (Predicting Score Gain)...")
    for method in methods:
        method_df = df[df['method'] == method].copy()
        
        if len(method_df) < 10:
            print(f"  ⚠️ Skipping {method}: Too few samples ({len(method_df)})")
            continue
            
        X = method_df[feature_cols]
        y = method_df['gain']
        
        # Simple split or CV
        model = lgb.LGBMRegressor(n_estimators=100, random_state=42, verbosity=-1)
        
        # Cross-Validation Predictions to evaluate "Truthfulness"
        kf = KFold(n_splits=min(5, len(method_df)), shuffle=True, random_state=42)
        y_pred_cv = cross_val_predict(model, X, y, cv=kf)
        
        # Train final model
        model.fit(X, y)
        joblib.dump(model, f"{MODEL_DIR}/regressor_{method}.pkl")
        
        # Metrics
        rmse = np.sqrt(mean_squared_error(y, y_pred_cv))
        
        # "Win" Classification Accuracy (Did we correctly predict positive gain?)
        actual_win = (y > 0)
        pred_win = (y_pred_cv > 0)
        win_acc = np.mean(actual_win == pred_win)
        
        results.append({
            'Method': method,
            'Samples': len(method_df),
            'RMSE': rmse,
            'Win_Pred_Acc': win_acc,
            'Avg_Gain': y.mean()
        })
        
        print(f"  ✅ {method:25s} | N={len(method_df):3d} | WinAcc={win_acc:.2%} | AvgGain={y.mean():.4f}")

    # Summary
    res_df = pd.DataFrame(results).sort_values('Win_Pred_Acc', ascending=False)
    print("\n📊 Regression Evaluation Summary:")
    print(res_df)
    res_df.to_csv("meta_regression_summary.csv", index=False)

if __name__ == "__main__":
    data = load_and_prepare_data()
    if data is not None:
        train_method_regressors(data)


📂 Loading data...
✅ Loaded 886 samples (110 datasets).
🧠 Meta-Features (37): ['Number_of_Instances', 'Number_of_Attributes', 'Number_of_Missing_Values', 'Percentage_of_Missing_Values', 'Number_of_Numeric_Attributes']...

🏋️ Training Regressors (Predicting Score Gain)...
  ✅ OneHotEncoder_LowCard     | N= 36 | WinAcc=47.22% | AvgGain=0.0008
  ✅ FrequencyEncoder          | N= 38 | WinAcc=60.53% | AvgGain=-0.0074
  ✅ LabelEncoder              | N= 38 | WinAcc=28.95% | AvgGain=0.0005
  ✅ RareLabelEncoder          | N= 38 | WinAcc=89.47% | AvgGain=-0.0019
  ✅ CategoricalInteractions   | N= 35 | WinAcc=42.86% | AvgGain=-0.0001
  ✅ StandardScaler            | N=101 | WinAcc=51.49% | AvgGain=-0.0007
  ✅ Binning_Quantile_10       | N=101 | WinAcc=62.38% | AvgGain=-0.0226
  ✅ ArithmeticInteractions    | N= 92 | WinAcc=46.74% | AvgGain=0.0014
  ✅ PCA_95                    | N= 91 | WinAcc=82.42% | AvgGain=-0.0222
  ✅ FastICA                   | N= 91 | WinAcc=56.04% | AvgGain=-0.0049
  ✅ SelectKB

## Classification Model Training 

In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os

# Configuration
META_FEATURES_FILE = "meta_features_large.csv"
PERFORMANCE_FILE = "performance_labels_large.csv"
MODEL_OUTPUT_FILE = "meta_model_lgbm_large.pkl"
ENCODER_OUTPUT_FILE = "meta_label_encoder_large.pkl"

def train_meta_model():
    print("🚀 Starting Large-Scale Meta-Model Training...")
    
    # 1. Load Data
    if not os.path.exists(META_FEATURES_FILE) or not os.path.exists(PERFORMANCE_FILE):
        print(f"❌ Missing data files. Please ensure '{META_FEATURES_FILE}' and '{PERFORMANCE_FILE}' exist.")
        return

    df_X = pd.read_csv(META_FEATURES_FILE)
    df_Y = pd.read_csv(PERFORMANCE_FILE)
    
    print(f"  Loaded X (Features): {df_X.shape}")
    print(f"  Loaded Y (Performance): {df_Y.shape}")

    # 2. Process Y to find the "Best Method"
    baseline_scores = df_Y[df_Y['method'] == 'Baseline'].groupby('dataset_id')['score'].mean()
    
    df_Y['baseline_score'] = df_Y['dataset_id'].map(baseline_scores)
    df_Y['delta'] = df_Y['score'] - df_Y['baseline_score']
    
    best_methods = []
    dataset_ids = df_Y['dataset_id'].unique()
    
    for ds_id in dataset_ids:
        subset = df_Y[df_Y['dataset_id'] == ds_id]
        candidates = subset[subset['method'] != 'Baseline']
        
        if candidates.empty:
            best_method = 'Baseline'
        else:
            best_candidate = candidates.loc[candidates['delta'].idxmax()]
            # Threshold: 0.001 (0.1%) improvement required to switch from Baseline
            if best_candidate['delta'] > 0.001: 
                best_method = best_candidate['method']
            else:
                best_method = 'Baseline'
                
        best_methods.append({'dataset_id': ds_id, 'best_method': best_method})
        
    df_labels = pd.DataFrame(best_methods)
    print(f"  Computed Best Methods for {len(df_labels)} datasets.")
    print("  Label Distribution:\n", df_labels['best_method'].value_counts())

    # 3. Merge X and Y
    df_train = pd.merge(df_X, df_labels, on='dataset_id', how='inner')
    
    drop_cols = ['dataset_id', 'dataset_name', 'best_method']
    X = df_train.drop(columns=drop_cols).select_dtypes(include=np.number)
    y_raw = df_train['best_method']
    
    # 4. Encode Labels
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    
    # 5. Train LightGBM
    print("\n🧠 Training LightGBM Classifier...")
    
    # Split for validation
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    except ValueError:
        print("⚠️ Warning: Some classes have too few members for stratification. Falling back to random split.")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    clf = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    )
    
    clf.fit(X_train, y_train)
    
    # 6. Evaluate
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"  ✅ Validation Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    target_names = [str(c) for c in le.classes_]
    print(classification_report(y_test, y_pred, target_names=target_names, labels=range(len(target_names)), zero_division=0))
    
    # 7. Save Model & Encoder
    joblib.dump(clf, MODEL_OUTPUT_FILE)
    joblib.dump(le, ENCODER_OUTPUT_FILE)
    print(f"\n💾 Model saved to: {MODEL_OUTPUT_FILE}")
    print(f"💾 Label Encoder saved to: {ENCODER_OUTPUT_FILE}")
    print("🎉 Meta-Learning Training Complete!")

if __name__ == "__main__":
    train_meta_model()


🚀 Starting Large-Scale Meta-Model Training...
  Loaded X (Features): (110, 39)
  Loaded Y (Performance): (886, 6)
  Computed Best Methods for 110 datasets.
  Label Distribution:
 best_method
Baseline                   41
Binning_Quantile_10        13
ArithmeticInteractions     12
SelectKBest_50pct          10
StandardScaler              8
PCA_95                      5
FastICA                     5
CategoricalInteractions     4
OneHotEncoder_LowCard       4
LabelEncoder                3
FrequencyEncoder            2
ImputeMedianMode            2
MissingIndicator            1
Name: count, dtype: int64

🧠 Training LightGBM Classifier...
⚠️ Warning: Some classes have too few members for stratification. Falling back to random split.
  ✅ Validation Accuracy: 0.3182

Classification Report:
                         precision    recall  f1-score   support

 ArithmeticInteractions       0.00      0.00      0.00         2
               Baseline       0.50      0.78      0.61         9
    Binnin

## inference

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from scipy.stats import skew, kurtosis, entropy

# ============================================================
# Configuration
# ============================================================
MODEL_DIR = "meta_models_regression"   # Regression models
CLASSIFICATION_MODEL = "meta_model_lgbm.pkl"
ENCODER_FILE = "meta_label_encoder.pkl"

# ============================================================
# Meta-Feature Extraction (70 features - matches training data)
# ============================================================

def get_quartiles(series):
    """Return Q1, Q2 (Median), Q3."""
    if len(series) == 0: 
        return 0, 0, 0
    return series.quantile([0.25, 0.5, 0.75]).tolist()

def calc_stats_full(series, prefix):
    """
    Calculate Min, Mean, Max, Std, Q1, Q2, Q3 for a series.
    This matches the FULL 70-feature extraction used in training.
    """
    if len(series) == 0:
        return {
            f'Min_{prefix}': 0, f'Mean_{prefix}': 0, f'Max_{prefix}': 0, f'Std_{prefix}': 0,
            f'Q1_{prefix}': 0, f'Q2_{prefix}': 0, f'Q3_{prefix}': 0
        }
    
    q1, q2, q3 = get_quartiles(series)
    return {
        f'Min_{prefix}': series.min(),
        f'Mean_{prefix}': series.mean(),
        f'Max_{prefix}': series.max(),
        f'Std_{prefix}': series.std(ddof=1) if len(series) > 1 else 0,
        f'Q1_{prefix}': q1,
        f'Q2_{prefix}': q2,
        f'Q3_{prefix}': q3
    }

def extract_meta_features_full(df, target_col=None, problem_type='classification'):
    """
    Extract 70 meta-features from a DataFrame.
    This matches the EXACT feature extraction used in training (importopenml.ipynb).
    
    Args:
        df: Input DataFrame
        target_col: Name of target column (will be excluded from feature analysis)
        problem_type: 'classification', 'binary', 'multiclass', or 'regression'
    
    Returns:
        DataFrame with one row of meta-features
    """
    features = {}
    
    # Separate X and y
    if target_col and target_col in df.columns:
        X = df.drop(columns=[target_col])
        y = df[target_col]
    else:
        X = df
        y = None
    
    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    
    n_instances = len(df)
    n_attrs = len(X.columns)
    
    # ========== Generic Features (49-61) ==========
    features['Number_of_Instances'] = n_instances
    features['Number_of_Attributes'] = n_attrs
    features['Dimensionality'] = n_attrs / n_instances if n_instances > 0 else 0
    
    missing_total = X.isnull().sum().sum()
    features['Number_of_Missing_Values'] = missing_total
    features['Percentage_of_Missing_Values'] = missing_total / (n_instances * n_attrs) if n_instances * n_attrs > 0 else 0
    
    rows_missing = X.isnull().any(axis=1).sum()
    features['Number_of_Instances_with_Missing_Values'] = rows_missing
    features['Percentage_of_Instances_with_Missing_Values'] = rows_missing / n_instances if n_instances > 0 else 0
    
    # Class-related features
    if y is not None and problem_type != 'regression':
        class_counts = y.value_counts()
        features['Number_of_Classes'] = len(class_counts)
        probs = class_counts / n_instances
        features['Class_Entropy'] = entropy(probs, base=2)
        features['Minority_Class_Size'] = class_counts.min()
        features['Majority_Class_Size'] = class_counts.max()
        features['Minority_Class_Percentage'] = class_counts.min() / n_instances
        features['Majority_Class_Percentage'] = class_counts.max() / n_instances
    else:
        features['Number_of_Classes'] = 0
        features['Class_Entropy'] = 0
        features['Minority_Class_Size'] = 0
        features['Majority_Class_Size'] = 0
        features['Minority_Class_Percentage'] = 0
        features['Majority_Class_Percentage'] = 0

    # ========== Continuous Attributes (1-26) ==========
    features['Number_of_Continuous_Attributes'] = len(num_cols)
    features['Percentage_of_Continuous_Attributes'] = len(num_cols) / n_attrs if n_attrs > 0 else 0
    
    if len(num_cols) > 0:
        X_num = X[num_cols]
        means = X_num.mean()
        stds = X_num.std()
        skews = X_num.apply(lambda x: skew(x.dropna()))
        kurtoses = X_num.apply(lambda x: kurtosis(x.dropna()))
        
        features.update(calc_stats_full(means, 'Means_of_Continuous_Attributes'))
        features.update(calc_stats_full(stds, 'Std_of_Continuous_Attributes'))
        features.update(calc_stats_full(skews, 'Skewness_of_Continuous_Attributes'))
        features.update(calc_stats_full(kurtoses, 'Kurtosis_of_Continuous_Attributes'))
    else:
        for stat in ['Means', 'Std', 'Skewness', 'Kurtosis']:
            features.update(calc_stats_full(pd.Series([]), f'{stat}_of_Continuous_Attributes'))

    # ========== Categorical & Binary Attributes (27-48) ==========
    binary_cols = [c for c in X.columns if X[c].nunique() == 2]
    features['Number_of_Categorical_Attributes'] = len(cat_cols)
    features['Number_of_Binary_Attributes'] = len(binary_cols)
    features['Percentage_of_Categorical_Attributes'] = len(cat_cols) / n_attrs if n_attrs > 0 else 0
    features['Percentage_of_Binary_Attributes'] = len(binary_cols) / n_attrs if n_attrs > 0 else 0
    
    if len(cat_cols) > 0:
        entropies = []
        mutual_infos = []  # Simplified: set to 0 for local inference
        distinct_values = []
        
        for c in cat_cols:
            col_data = X[c].astype(str)
            # Entropy
            vc = col_data.value_counts(normalize=True)
            entropies.append(entropy(vc, base=2))
            # Distinct Values
            distinct_values.append(col_data.nunique())
            # Mutual Information (simplified: set to 0 for speed)
            mutual_infos.append(0)

        features.update(calc_stats_full(pd.Series(entropies), 'Attribute_Entropy'))
        features.update(calc_stats_full(pd.Series(mutual_infos), 'Mutual_Information'))
        features.update(calc_stats_full(pd.Series(distinct_values), 'Attribute_Distinct_Values'))
        
        # Equivalent Number of Attributes & Noise to Signal Ratio
        features['Equivalent_Number_of_Attributes'] = 0
        features['Noise_to_Signal_Ratio'] = 0
        
    else:
        for stat in ['Attribute_Entropy', 'Mutual_Information', 'Attribute_Distinct_Values']:
            features.update(calc_stats_full(pd.Series([]), stat))
        features['Equivalent_Number_of_Attributes'] = 0
        features['Noise_to_Signal_Ratio'] = 0

    return pd.DataFrame([features])

# ============================================================
# Regression-based Recommendation (Recommended!)
# ============================================================

def recommend_preprocessing_regression(df, target_col=None, problem_type='classification'):
    """
    Use regression models to predict Score Gain for each preprocessing method.
    This is the RECOMMENDED approach.
    
    Returns:
        DataFrame with predicted gains for each method
    """
    print("🤖 Meta-Model Inference (Regression)...")
    
    if not os.path.exists(MODEL_DIR):
        print(f"❌ Model directory '{MODEL_DIR}' not found. Please train models first.")
        return None
    
    # 1. Extract Features
    print("   Extracting meta-features...")
    X_meta = extract_meta_features_full(df, target_col, problem_type)
    print(f"   ✅ Extracted {len(X_meta.columns)} features")
    
    # 2. Load and run each regressor
    predictions = []
    
    for model_file in os.listdir(MODEL_DIR):
        if not model_file.startswith('regressor_') or not model_file.endswith('.pkl'):
            continue
            
        method = model_file.replace('regressor_', '').replace('.pkl', '')
        
        try:
            model = joblib.load(os.path.join(MODEL_DIR, model_file))
            
            # Align features with what model expects
            feature_cols = model.feature_name_
            
            # Handle missing columns (fill with 0)
            for col in feature_cols:
                if col not in X_meta.columns:
                    X_meta[col] = 0
            
            X = X_meta[feature_cols]
            pred_gain = model.predict(X)[0]
            
            predictions.append({
                'Method': method,
                'Predicted_Gain': pred_gain,
                'Recommended': pred_gain > 0
            })
        except Exception as e:
            print(f"   ⚠️ Error loading {method}: {e}")
            continue
    
    if not predictions:
        print("❌ No predictions made. Check if models exist.")
        return None
    
    result = pd.DataFrame(predictions).sort_values('Predicted_Gain', ascending=False)
    
    print(f"\n📊 Preprocessing Recommendations:")
    print("-" * 50)
    for _, row in result.head(5).iterrows():
        status = "✅" if row['Recommended'] else "❌"
        print(f"   {status} {row['Method']:25s} | Gain: {row['Predicted_Gain']:+.4f}")
    print("-" * 50)
    
    recommended = result[result['Recommended']]
    if len(recommended) > 0:
        print(f"\n🏆 Top Recommended: {recommended.iloc[0]['Method']}")
    else:
        print("\n⚠️ No method predicted to improve performance. Use Baseline.")
    
    return result

# ============================================================
# Classification-based Recommendation (Legacy)
# ============================================================

def recommend_preprocessing_classification(df, target_col=None):
    """
    Use classification model to predict the "best" preprocessing method.
    NOTE: This approach has lower accuracy (~32%). Use regression instead.
    """
    print("🤖 Meta-Model Inference (Classification - Legacy)...")
    
    if not os.path.exists(CLASSIFICATION_MODEL) or not os.path.exists(ENCODER_FILE):
        print("❌ Model files not found. Please train the model first.")
        return None
        
    clf = joblib.load(CLASSIFICATION_MODEL)
    le = joblib.load(ENCODER_FILE)
    
    print("   Extracting meta-features...")
    X_meta = extract_meta_features_full(df, target_col)
    
    try:
        probs = clf.predict_proba(X_meta)[0]
        best_idx = np.argmax(probs)
        best_method = le.inverse_transform([best_idx])[0]
        confidence = probs[best_idx]
        
        print(f"   ✅ Prediction Complete.")
        print(f"   🏆 Recommended Method: {best_method} (Confidence: {confidence:.2%})")
        
        return best_method
        
    except Exception as e:
        print(f"❌ Prediction failed: {e}")
        return None

# ============================================================
# Main Function
# ============================================================

def suggest_preprocessing(df, target_col=None, method='regression'):
    """
    Main entry point for preprocessing recommendation.
    
    Args:
        df: Input DataFrame
        target_col: Name of target column
        method: 'regression' (recommended) or 'classification' (legacy)
    
    Returns:
        Recommended preprocessing method(s)
    """
    if method == 'regression':
        return recommend_preprocessing_regression(df, target_col)
    else:
        return recommend_preprocessing_classification(df, target_col)

# ============================================================
# Example Usage
# ============================================================
if __name__ == "__main__":
    # Create a dummy dataframe for testing
    df_test = pd.DataFrame({
        'A': np.random.rand(100),
        'B': np.random.rand(100) * 100,
        'C': np.random.choice(['a', 'b', 'c'], 100),
        'target': np.random.randint(0, 2, 100)
    })
    
    # Use regression-based recommendation (default)
    result = suggest_preprocessing(df_test, target_col='target', method='regression')


🤖 Meta-Model Inference (Regression)...
   Extracting meta-features...
   ✅ Extracted 70 features

📊 Preprocessing Recommendations:
--------------------------------------------------
   ✅ MissingIndicator          | Gain: +0.0009
   ✅ OneHotEncoder_LowCard     | Gain: +0.0008
   ✅ LabelEncoder              | Gain: +0.0005
   ✅ ArithmeticInteractions    | Gain: +0.0003
   ❌ CategoricalInteractions   | Gain: -0.0001
--------------------------------------------------

🏆 Top Recommended: MissingIndicator


# OLD Version


## Collect performance Labels

In [ ]:
import openml
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import LabelEncoder
import preprocessing_function as pf
import os
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Configuration
SUITE_ID = 99  # OpenML-CC18
RESULTS_FILE = "performance_labels.csv"

def get_model(problem_type):
    """Return LightGBM model with default parameters."""
    common_params = {
        "n_estimators": 100,
        "verbosity": -1,
        "n_jobs": 1,
        "random_state": 42
    }
    if problem_type == 'regression':
        return lgb.LGBMRegressor(objective='regression', **common_params)
    else:
        return lgb.LGBMClassifier(objective='binary' if problem_type == 'binary' else 'multiclass', **common_params)

def evaluate_dataset(df, target_col, problem_type):
    """Evaluate dataset using 5-Fold CV with LightGBM."""
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # Basic encoding for LightGBM to handle object types
    for col in X.select_dtypes(include=['object']).columns:
        X[col] = X[col].astype('category')
        
    if problem_type != 'regression' and y.dtype == 'object':
        le = LabelEncoder()
        y = le.fit_transform(y)

    model = get_model(problem_type)
    
    if problem_type == 'regression':
        cv = KFold(n_splits=5, shuffle=True, random_state=42)
        scoring = 'neg_root_mean_squared_error'
    else:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scoring = 'accuracy'
        
    try:
        scores = cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=1)
        return np.mean(scores)
    except Exception as e:
        return np.nan

def run_performance_collection():
    print(f"🚀 Starting Comprehensive Performance Label Collection on OpenML Suite {SUITE_ID}")
    
    if not os.path.exists(RESULTS_FILE):
        pd.DataFrame(columns=[
            'dataset_id', 'dataset_name', 'problem_type', 
            'method', 'score', 'is_baseline'
        ]).to_csv(RESULTS_FILE, index=False)

    suite = openml.study.get_suite(SUITE_ID)
    tasks = suite.tasks
    
    for i, task_id in enumerate(tasks):
        try:
            print(f"\nProcessing Task {task_id} ({i+1}/{len(tasks)})...")
            task = openml.tasks.get_task(task_id, download_data=False)
            dataset = openml.datasets.get_dataset(task.dataset_id, download_data=True)
            
            df, _, _, _ = dataset.get_data(dataset_format="dataframe")
            target_col = task.target_name
            
            # Determine problem type
            n_classes = task.class_labels
            if n_classes is None:
                problem_type = 'regression'
            elif len(n_classes) == 2:
                problem_type = 'binary'
            else:
                problem_type = 'multiclass'
            
            print(f"  Dataset: {dataset.name} | Type: {problem_type} | Shape: {df.shape}")
            
            # Skip large datasets to prevent timeouts (e.g. MNIST)
            if df.shape[0] > 10000:
                print(f"⚠️ Skipping {dataset.name} (Too large: {df.shape})")
                continue
            
            results_batch = []
            
            # --- 1. Baseline ---
            baseline_score = evaluate_dataset(df.copy(), target_col, problem_type)
            print(f"  Baseline Score: {baseline_score:.4f}")
            results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'Baseline', 'score': baseline_score, 'is_baseline': True})
            
            if np.isnan(baseline_score): continue

            num_cols = df.select_dtypes(include=np.number).columns.tolist()
            if target_col in num_cols: num_cols.remove(target_col)
            cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
            if target_col in cat_cols: cat_cols.remove(target_col)

            # --- 2. Categorical Methods ---
            if cat_cols:
                # OHE (<10 categories)
                low_card_cols = [c for c in cat_cols if df[c].nunique() < 10]
                if low_card_cols:
                    try:
                        df_ohe = pf.one_hot_encode(df.copy(), columns=low_card_cols)
                        score = evaluate_dataset(df_ohe, target_col, problem_type)
                        results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'OneHotEncoder_LowCard', 'score': score, 'is_baseline': False})
                    except: pass

                # Frequency Encoding
                try:
                    df_freq = pf.frequency_encode(df.copy(), columns=cat_cols)
                    score = evaluate_dataset(df_freq, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'FrequencyEncoder', 'score': score, 'is_baseline': False})
                except: pass

                # Label Encoding
                try:
                    df_le = pf.label_encode(df.copy(), columns=cat_cols)
                    score = evaluate_dataset(df_le, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'LabelEncoder', 'score': score, 'is_baseline': False})
                except: pass

                # Target Encoding (Manual Implementation)
                try:
                    df_te = df.copy()
                    # Simple Target Encoding with smoothing
                    if problem_type != 'regression':
                        y_enc = LabelEncoder().fit_transform(df[target_col])
                    else:
                        y_enc = df[target_col]
                    
                    for c in cat_cols:
                        global_mean = y_enc.mean()
                        agg = df_te.groupby(c)[target_col].agg(['count', 'mean'])
                        counts = agg['count']
                        means = agg['mean']
                        smooth = 10
                        smooth_means = (counts * means + smooth * global_mean) / (counts + smooth)
                        df_te[c] = df_te[c].map(smooth_means)
                    
                    score = evaluate_dataset(df_te, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'TargetEncoder', 'score': score, 'is_baseline': False})
                except: pass

                # Rare Label Encoding (Manual)
                try:
                    df_rare = df.copy()
                    for c in cat_cols:
                        vc = df_rare[c].value_counts(normalize=True)
                        rare = vc[vc < 0.05].index
                        df_rare[c] = df_rare[c].replace(rare, 'Other')
                    score = evaluate_dataset(df_rare, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'RareLabelEncoder', 'score': score, 'is_baseline': False})
                except: pass

            # --- 3. Numerical Methods ---
            if num_cols:
                # Log Transform (Yeo-Johnson)
                try:
                    df_log = pf.apply_power_transform(df.copy(), column=num_cols, method='yeo-johnson')
                    score = evaluate_dataset(df_log, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'LogTransform_YeoJohnson', 'score': score, 'is_baseline': False})
                except: pass
                
            # --- [NEW] 3.1 Scaling ---
            if num_cols:
                try:
                    from sklearn.preprocessing import StandardScaler
                    df_scaled = df.copy()
                    df_scaled[num_cols] = StandardScaler().fit_transform(df_scaled[num_cols])
                    score = evaluate_dataset(df_scaled, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'StandardScaler', 'score': score, 'is_baseline': False})
                except: pass

            # --- [NEW] 3.2 Binning (Discretization) ---
            if num_cols:
                try:
                    from sklearn.preprocessing import KBinsDiscretizer
                    df_bin = df.copy()
                    # Use 10 bins, ordinal encoding
                    est = KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='quantile')
                    # Fill NaNs before binning to avoid errors
                    df_bin[num_cols] = df_bin[num_cols].fillna(df_bin[num_cols].median())
                    df_bin[num_cols] = est.fit_transform(df_bin[num_cols])
                    score = evaluate_dataset(df_bin, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'Binning_Quantile_10', 'score': score, 'is_baseline': False})
                except: pass
            
            # --- [NEW] 7.2 PCA ---
            if num_cols and len(num_cols) > 2:
                try:
                    from sklearn.decomposition import PCA
                    df_pca = df.copy()
                    # Fill NaNs first
                    X_num = df_pca[num_cols].fillna(df_pca[num_cols].median())
                    # Keep 95% variance
                    pca = PCA(n_components=0.95)
                    X_pca = pca.fit_transform(X_num)
                    
                    # Reconstruct DataFrame
                    df_pca_new = pd.DataFrame(X_pca, index=df.index)
                    # Add back categorical/target
                    other_cols = [c for c in df.columns if c not in num_cols]
                    df_pca = pd.concat([df_pca_new, df[other_cols]], axis=1)
                    
                    score = evaluate_dataset(df_pca, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'PCA_95', 'score': score, 'is_baseline': False})
                except: pass

            # --- [NEW] 7.3 Feature Selection (SelectKBest) ---
            if num_cols and len(num_cols) > 10:
                try:
                    from sklearn.feature_selection import SelectKBest, f_classif, f_regression
                    df_fs = df.copy()
                    X_fs = df_fs.drop(columns=[target_col])
                    y_fs = df_fs[target_col]
                    
                    # Handle missing/categorical for selection
                    X_fs_num = X_fs[num_cols].fillna(X_fs[num_cols].median())
                    
                    if problem_type == 'regression':
                        score_func = f_regression
                    else:
                        score_func = f_classif
                        if y_fs.dtype == 'object':
                            y_fs = LabelEncoder().fit_transform(y_fs)
                            
                    # Select top 50% features
                    k = int(len(num_cols) * 0.5)
                    selector = SelectKBest(score_func=score_func, k=k)
                    selector.fit(X_fs_num, y_fs)
                    
                    selected_cols = [num_cols[i] for i in selector.get_support(indices=True)]
                    # Keep selected num cols + all cat cols + target
                    keep_cols = selected_cols + cat_cols + [target_col]
                    df_fs = df_fs[keep_cols]
                    
                    score = evaluate_dataset(df_fs, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'SelectKBest_50pct', 'score': score, 'is_baseline': False})
                except: pass

            # --- 4. Missing Values ---
            missing_cols = [c for c in df.columns if df[c].isnull().any() and c != target_col]
            if missing_cols:
                # Impute Median/Mode
                try:
                    df_imp = df.copy()
                    num_miss = [c for c in missing_cols if c in num_cols]
                    cat_miss = [c for c in missing_cols if c in cat_cols]
                    if num_miss: df_imp = pf.impute_median(df_imp, num_miss)
                    if cat_miss: df_imp = pf.impute_mode(df_imp, cat_miss)
                    score = evaluate_dataset(df_imp, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'ImputeMedianMode', 'score': score, 'is_baseline': False})
                except: pass

                # Impute Constant
                try:
                    df_const = pf.impute_constant(df.copy(), columns=missing_cols, fill_value=-999)
                    score = evaluate_dataset(df_const, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'ImputeConstant', 'score': score, 'is_baseline': False})
                except: pass

                # Missing Indicator
                try:
                    df_ind = pf.add_missing_indicator(df.copy(), columns=missing_cols)
                    score = evaluate_dataset(df_ind, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'MissingIndicator', 'score': score, 'is_baseline': False})
                except: pass

            # --- 5. Interactions ---
            if len(num_cols) >= 2:
                try:
                    # Arithmetic Interactions
                    df_inter = pf.create_features_from_correlation_analysis(
                        df.copy(), 
                        correlation_threshold=0.7, # Increased from 0.1 to avoid explosion
                        feature_types=['product', 'ratio', 'sum', 'difference'],
                        max_new_features=50 # Strict limit
                    )
                    score = evaluate_dataset(df_inter, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'ArithmeticInteractions', 'score': score, 'is_baseline': False})
                except: pass

            if len(cat_cols) >= 2:
                try:
                    # Categorical Interactions
                    df_cat_inter = pf.combine_categorical_features(
                        df.copy(), 
                        columns_to_combine=cat_cols[:2], # Just try first pair for test
                        new_col_name='combined_cat',
                        drop_original=False
                    )
                    score = evaluate_dataset(df_cat_inter, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'CategoricalInteractions', 'score': score, 'is_baseline': False})
                except: pass

            # --- 6. Date Features ---
            # Check for potential date columns (simple heuristic)
            date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
            if date_cols:
                try:
                    df_date = df.copy()
                    for dc in date_cols:
                        try:
                            df_date = pf.extract_datetime_features(df_date, dc)
                        except: pass
                    score = evaluate_dataset(df_date, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'DateFeatures', 'score': score, 'is_baseline': False})
                except: pass

            # --- 7. Dimensionality Reduction ---
            if num_cols:
                # FastICA
                try:
                    df_ica = pf.apply_fastica(df.copy(), replace_ratio=0.4, random_state=42)
                    score = evaluate_dataset(df_ica, target_col, problem_type)
                    results_batch.append({'dataset_id': task.dataset_id, 'dataset_name': dataset.name, 'problem_type': problem_type, 'method': 'FastICA', 'score': score, 'is_baseline': False})
                except: pass

            # Save Batch
            if results_batch:
                pd.DataFrame(results_batch).to_csv(RESULTS_FILE, mode='a', header=False, index=False)
                print(f"  ✅ Recorded {len(results_batch)} results.")
                
        except Exception as e:
            print(f"❌ Error processing task {task_id}: {e}")
            continue

if __name__ == "__main__":
    run_performance_collection()


🚀 Starting Comprehensive Performance Label Collection on OpenML Suite 99

Processing Task 3 (1/72)...
  Dataset: kr-vs-kp | Type: binary | Shape: (3196, 37)
  Baseline Score: 0.9941
Created new combined feature: 'combined_cat'
  ✅ Recorded 6 results.

Processing Task 6 (2/72)...
  Dataset: letter | Type: multiclass | Shape: (20000, 17)
⚠️ Skipping letter (Too large: (20000, 17))

Processing Task 11 (3/72)...
  Dataset: balance-scale | Type: multiclass | Shape: (625, 5)
  Baseline Score: 0.8640
⏳ [Feature Gen] Calculating correlation matrix...
ℹ️ [Feature Gen] No pairs found with correlation > 0.7. Skipping.
⚠️ n_components (4) >= n_features (4). Using 3
✅ FastICA (Optuna-Tuned Hybrid Mode):
   Replace ratio: 40.00% (from Optuna tuning)
   Replaced 1 features, kept 3, added 3 ICA components
   Total features: 5 → 7
   Replaced: ['left-distance']
   Kept: ['left-weight', 'right-distance', 'right-weight']
   Added 3 ICA interaction features
  ✅ Recorded 6 results.

Processing Task 12 (4/7

⏳ [Feature Gen] Calculating correlation matrix...
✓ [Feature Gen] Found 800 high correlation pairs.
✓ [Feature Gen] Generated 3200 raw candidate features.
  Basic Filter: 3200 → 3187 features
    Dropped: low_cardinality: 13


## meta features extract

In [2]:
import openml
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis, entropy
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
import os
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Configuration
SUITE_ID = 99  # OpenML-CC18
RESULTS_FILE = "meta_features_61.csv"

def get_quartiles(series):
    """Return Q1, Q2 (Median), Q3."""
    if len(series) == 0: return 0, 0, 0
    return series.quantile([0.25, 0.5, 0.75]).tolist()

def calc_stats(series, prefix):
    """Calculate Min, Mean, Max, Std, Q1, Q2, Q3 for a series of values."""
    if len(series) == 0:
        return {
            f'Min_{prefix}': 0, f'Mean_{prefix}': 0, f'Max_{prefix}': 0, f'Std_{prefix}': 0,
            f'Q1_{prefix}': 0, f'Q2_{prefix}': 0, f'Q3_{prefix}': 0
        }
    
    q1, q2, q3 = get_quartiles(series)
    return {
        f'Min_{prefix}': series.min(),
        f'Mean_{prefix}': series.mean(),
        f'Max_{prefix}': series.max(),
        f'Std_{prefix}': series.std(ddof=1) if len(series) > 1 else 0,
        f'Q1_{prefix}': q1,
        f'Q2_{prefix}': q2,
        f'Q3_{prefix}': q3
    }

def extract_meta_features(df, target_col, problem_type):
    features = {}
    
    # Separate columns
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    num_cols = X.select_dtypes(include=np.number).columns
    cat_cols = X.select_dtypes(include=['object', 'category']).columns
    
    n_instances = len(df)
    n_attrs = len(X.columns)
    
    # --- Generic (49-61) ---
    features['Number_of_Instances'] = n_instances
    features['Number_of_Attributes'] = n_attrs
    features['Dimensionality'] = n_attrs / n_instances if n_instances > 0 else 0
    
    missing_total = X.isnull().sum().sum()
    features['Number_of_Missing_Values'] = missing_total
    features['Percentage_of_Missing_Values'] = missing_total / (n_instances * n_attrs) if n_instances * n_attrs > 0 else 0
    
    rows_missing = X.isnull().any(axis=1).sum()
    features['Number_of_Instances_with_Missing_Values'] = rows_missing
    features['Percentage_of_Instances_with_Missing_Values'] = rows_missing / n_instances if n_instances > 0 else 0
    
    if problem_type != 'regression':
        class_counts = y.value_counts()
        features['Number_of_Classes'] = len(class_counts)
        probs = class_counts / n_instances
        features['Class_Entropy'] = entropy(probs, base=2)
        features['Minority_Class_Size'] = class_counts.min()
        features['Majority_Class_Size'] = class_counts.max()
        features['Minority_Class_Percentage'] = class_counts.min() / n_instances
        features['Majority_Class_Percentage'] = class_counts.max() / n_instances
    else:
        features['Number_of_Classes'] = 0
        features['Class_Entropy'] = 0
        features['Minority_Class_Size'] = 0
        features['Majority_Class_Size'] = 0
        features['Minority_Class_Percentage'] = 0
        features['Majority_Class_Percentage'] = 0

    # --- Continuous Attributes (1-26) ---
    features['Number_of_Continuous_Attributes'] = len(num_cols)
    features['Percentage_of_Continuous_Attributes'] = len(num_cols) / n_attrs if n_attrs > 0 else 0
    
    if len(num_cols) > 0:
        X_num = X[num_cols]
        means = X_num.mean()
        stds = X_num.std()
        skews = X_num.apply(lambda x: skew(x.dropna()))
        kurtoses = X_num.apply(lambda x: kurtosis(x.dropna()))
        
        features.update(calc_stats(means, 'Means_of_Continuous_Attributes'))
        features.update(calc_stats(stds, 'Std_of_Continuous_Attributes'))
        features.update(calc_stats(skews, 'Skewness_of_Continuous_Attributes'))
        features.update(calc_stats(kurtoses, 'Kurtosis_of_Continuous_Attributes'))
    else:
        for stat in ['Means', 'Std', 'Skewness', 'Kurtosis']:
            features.update(calc_stats([], f'{stat}_of_Continuous_Attributes'))

    # --- Categorical & Binary Attributes (27-48) ---
    # Identify binary (nunique=2)
    binary_cols = [c for c in X.columns if X[c].nunique() == 2]
    features['Number_of_Categorical_Attributes'] = len(cat_cols)
    features['Number_of_Binary_Attributes'] = len(binary_cols)
    features['Percentage_of_Categorical_Attributes'] = len(cat_cols) / n_attrs if n_attrs > 0 else 0
    features['Percentage_of_Binary_Attributes'] = len(binary_cols) / n_attrs if n_attrs > 0 else 0
    
    if len(cat_cols) > 0:
        entropies = []
        mutual_infos = []
        distinct_values = []
        
        # Pre-encode target for MI calculation
        y_encoded = y.astype('category').cat.codes if problem_type != 'regression' else y
        
        for c in cat_cols:
            col_data = X[c].astype(str)
            # Entropy
            vc = col_data.value_counts(normalize=True)
            entropies.append(entropy(vc, base=2))
            # Distinct Values
            distinct_values.append(col_data.nunique())
            
            # Mutual Information (simplified: skip if too slow, but requested)
            # Handling missing for MI: fill with mode
            try:
                col_filled = X[c].fillna(X[c].mode()[0] if not X[c].mode().empty else 'missing')
                le_col = col_filled.astype('category').cat.codes
                if problem_type == 'regression':
                    mi = mutual_info_regression(le_col.values.reshape(-1, 1), y_encoded, discrete_features=True)[0]
                else:
                    mi = mutual_info_classif(le_col.values.reshape(-1, 1), y_encoded, discrete_features=True)[0]
                mutual_infos.append(mi)
            except:
                mutual_infos.append(0)

        features.update(calc_stats(pd.Series(entropies), 'Attribute_Entropy'))
        features.update(calc_stats(pd.Series(mutual_infos), 'Mutual_Information'))
        features.update(calc_stats(pd.Series(distinct_values), 'Attribute_Distinct_Values'))
        
        # Equivalent Number of Attributes (EN_attr = H(C) / Mean(MI(C, A)))
        mean_mi = np.mean(mutual_infos) if mutual_infos else 0
        class_ent = features.get('Class_Entropy', 0)
        features['Equivalent_Number_of_Attributes'] = class_ent / mean_mi if mean_mi > 1e-5 else 0
        
        # Noise to Signal Ratio ( (Mean(H(A)) - Mean(MI)) / Mean(MI) )
        mean_ent = np.mean(entropies) if entropies else 0
        features['Noise_to_Signal_Ratio'] = (mean_ent - mean_mi) / mean_mi if mean_mi > 1e-5 else 0
        
    else:
        for stat in ['Attribute_Entropy', 'Mutual_Information', 'Attribute_Distinct_Values']:
            features.update(calc_stats([], stat))
        features['Equivalent_Number_of_Attributes'] = 0
        features['Noise_to_Signal_Ratio'] = 0

    return features

def run_collection():
    print(f"🚀 Starting Meta-Feature Collection (61 Features) on OpenML Suite {SUITE_ID}")
    
    suite = openml.study.get_suite(SUITE_ID)
    tasks = suite.tasks
    
    all_results = []
    
    for i, task_id in enumerate(tasks):
        try:
            print(f"Processing Task {task_id} ({i+1}/{len(tasks)})...")
            task = openml.tasks.get_task(task_id, download_data=False)
            dataset = openml.datasets.get_dataset(task.dataset_id, download_data=True)
            
            df, _, _, _ = dataset.get_data(dataset_format="dataframe")
            target_col = task.target_name
            
            # Determine problem type
            n_classes = task.class_labels
            if n_classes is None:
                problem_type = 'regression'
            elif len(n_classes) == 2:
                problem_type = 'binary'
            else:
                problem_type = 'multiclass'
            
            # Extract Features
            meta_feats = extract_meta_features(df, target_col, problem_type)
            meta_feats['dataset_id'] = task.dataset_id
            meta_feats['dataset_name'] = dataset.name
            
            all_results.append(meta_feats)
            
        except Exception as e:
            print(f"❌ Error processing task {task_id}: {e}")
            continue
            
    # Save to CSV
    if all_results:
        result_df = pd.DataFrame(all_results)
        # Reorder columns to put ID/Name first
        cols = ['dataset_id', 'dataset_name'] + [c for c in result_df.columns if c not in ['dataset_id', 'dataset_name']]
        result_df = result_df[cols]
        result_df.to_csv(RESULTS_FILE, index=False)
        print(f"\n✅ Collection Complete! Saved {len(result_df)} datasets to {RESULTS_FILE}")
    else:
        print("\n⚠️ No results collected.")

run_collection()

🚀 Starting Meta-Feature Collection (61 Features) on OpenML Suite 99
Processing Task 3 (1/72)...
Processing Task 6 (2/72)...
Processing Task 11 (3/72)...
Processing Task 12 (4/72)...
Processing Task 14 (5/72)...
Processing Task 15 (6/72)...
Processing Task 16 (7/72)...
Processing Task 18 (8/72)...
Processing Task 22 (9/72)...
Processing Task 23 (10/72)...
Processing Task 28 (11/72)...
Processing Task 29 (12/72)...
Processing Task 31 (13/72)...
Processing Task 32 (14/72)...
Processing Task 37 (15/72)...
Processing Task 43 (16/72)...
Processing Task 45 (17/72)...
Processing Task 49 (18/72)...
Processing Task 53 (19/72)...
Processing Task 219 (20/72)...
Processing Task 2074 (21/72)...
Processing Task 2079 (22/72)...
Processing Task 3021 (23/72)...
Processing Task 3022 (24/72)...
Processing Task 3481 (25/72)...
Processing Task 3549 (26/72)...
Processing Task 3560 (27/72)...
Processing Task 3573 (28/72)...
Processing Task 3902 (29/72)...
Processing Task 3903 (30/72)...
Processing Task 3904 (

## train meta model

### classification

In [5]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os

# Configuration
META_FEATURES_FILE = "meta_features_61.csv"
PERFORMANCE_FILE = "performance_labels.csv"
MODEL_OUTPUT_FILE = "meta_model_lgbm.pkl"
ENCODER_OUTPUT_FILE = "meta_label_encoder.pkl"

def train_meta_model():
    print("🚀 Starting Meta-Model Training with LightGBM...")
    
    # 1. Load Data
    if not os.path.exists(META_FEATURES_FILE) or not os.path.exists(PERFORMANCE_FILE):
        print(f"❌ Missing data files. Please ensure '{META_FEATURES_FILE}' and '{PERFORMANCE_FILE}' exist.")
        return

    df_X = pd.read_csv(META_FEATURES_FILE)
    df_Y = pd.read_csv(PERFORMANCE_FILE)
    
    print(f"  Loaded X (Features): {df_X.shape}")
    print(f"  Loaded Y (Performance): {df_Y.shape}")

    # 2. Process Y to find the "Best Method" for each dataset
    # We need to pivot or group by dataset_id to compare scores
    
    # Get Baseline Score for each dataset
    # Handle duplicates by taking the mean score for Baseline
    baseline_scores = df_Y[df_Y['method'] == 'Baseline'].groupby('dataset_id')['score'].mean()
    
    # Calculate Delta (Improvement over Baseline)
    df_Y['baseline_score'] = df_Y['dataset_id'].map(baseline_scores)
    df_Y['delta'] = df_Y['score'] - df_Y['baseline_score']
    
    # Strategy: Find method with Max Delta
    # If Max Delta <= 0 (or very small), then 'Baseline' is the best choice.
    
    best_methods = []
    dataset_ids = df_Y['dataset_id'].unique()
    
    for ds_id in dataset_ids:
        subset = df_Y[df_Y['dataset_id'] == ds_id]
        
        # Filter out Baseline row for comparison (we want to see if any candidate beats it)
        candidates = subset[subset['method'] != 'Baseline']
        
        if candidates.empty:
            best_method = 'Baseline'
        else:
            best_candidate = candidates.loc[candidates['delta'].idxmax()]
            if best_candidate['delta'] > 0.001: # Threshold for "significant" improvement
                best_method = best_candidate['method']
            else:
                best_method = 'Baseline'
                
        best_methods.append({'dataset_id': ds_id, 'best_method': best_method})
        
    df_labels = pd.DataFrame(best_methods)
    print(f"  Computed Best Methods for {len(df_labels)} datasets.")
    print("  Label Distribution:\n", df_labels['best_method'].value_counts())

    # 3. Merge X and Y
    # Ensure we only train on datasets where we have both features and labels
    df_train = pd.merge(df_X, df_labels, on='dataset_id', how='inner')
    
    # Drop ID and Name columns from training features
    drop_cols = ['dataset_id', 'dataset_name', 'best_method']
    # Also drop any non-numeric columns if they slipped in (though meta_features should be all numeric)
    X = df_train.drop(columns=drop_cols).select_dtypes(include=np.number)
    y_raw = df_train['best_method']
    
    # 4. Encode Labels
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    
    # 5. Train LightGBM
    print("\n🧠 Training LightGBM Classifier...")
    
    # Split for validation
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    except ValueError:
        print("⚠️ Warning: Some classes have too few members for stratification. Falling back to random split.")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    clf = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    )
    
    clf.fit(X_train, y_train)
    
    # 6. Evaluate
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"  ✅ Validation Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    # Map classes back to names for report
    target_names = [str(c) for c in le.classes_]
    print(classification_report(y_test, y_pred, target_names=target_names, labels=range(len(target_names)), zero_division=0))
    
    # 7. Feature Importance Analysis
    print("\n📊 Feature Importance Analysis:")
    importance = clf.feature_importances_
    feature_names = X.columns
    
    #Create a DataFrame for better visualization
    feature_imp = pd.DataFrame(sorted(zip(importance, feature_names)), columns=['Value','Feature']) 
    feature_imp = feature_imp.sort_values(by="Value", ascending=False).head(20)
    


### regression

In [6]:
#OLD version
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error
import joblib
import os

LABELS_FILE = "performance_labels.csv"      # old version
META_FEATURES_FILE = "meta_features_61.csv" # old version
MODEL_DIR = "meta_models_regression"

os.makedirs(MODEL_DIR, exist_ok=True)

def load_and_prepare_data():
    print("📂 Loading OLD version data (better coverage)...")
    
    df_lab = pd.read_csv(LABELS_FILE)
    df_meta = pd.read_csv(META_FEATURES_FILE)
    
    df_lab = df_lab.dropna(subset=['score'])
    
    # Baseline 
    baseline_scores = df_lab[df_lab['method'] == 'Baseline'].groupby('dataset_id')['score'].first()
    
    df_lab['baseline_score'] = df_lab['dataset_id'].map(baseline_scores)
    df_lab['gain'] = df_lab['score'] - df_lab['baseline_score']
    df_lab = df_lab.dropna(subset=['gain'])
    
    # merge
    full_df = pd.merge(df_lab, df_meta, on='dataset_id', how='inner')
    
    print(f"✅ Loaded {len(full_df)} samples ({full_df['dataset_id'].nunique()} datasets)")
    return full_df

def train_method_regressors(df):
    metadata_cols = ['dataset_id', 'dataset_name', 'dataset_name_x', 'dataset_name_y', 
                     'problem_type', 'method', 'score', 'is_baseline', 'baseline_score', 'gain']
    feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in metadata_cols]
    
    print(f"🧠 Meta-Features ({len(feature_cols)}): {feature_cols[:5]}...")
    
    methods = [m for m in df['method'].unique() if m != 'Baseline']
    results = []
    
    print("\n🏋️ Training Regressors (Predicting Score Gain)...")
    for method in methods:
        method_df = df[df['method'] == method].copy()
        
        if len(method_df) < 10:
            print(f"  ⚠️ Skipping {method}: Too few samples ({len(method_df)})")
            continue
            
        X = method_df[feature_cols]
        y = method_df['gain']
        
        model = lgb.LGBMRegressor(n_estimators=100, random_state=42, verbosity=-1)
        
        kf = KFold(n_splits=min(5, len(method_df)), shuffle=True, random_state=42)
        y_pred_cv = cross_val_predict(model, X, y, cv=kf)
        
        model.fit(X, y)
        joblib.dump(model, f"{MODEL_DIR}/regressor_{method}.pkl")
        
        rmse = np.sqrt(mean_squared_error(y, y_pred_cv))
        
        actual_win = (y > 0)
        pred_win = (y_pred_cv > 0)
        win_acc = np.mean(actual_win == pred_win)
        
        results.append({
            'Method': method,
            'Samples': len(method_df),
            'RMSE': rmse,
            'Win_Pred_Acc': win_acc,
            'Avg_Gain': y.mean()
        })
        
        print(f"  ✅ {method:25s} | N={len(method_df):3d} | WinAcc={win_acc:.2%} | AvgGain={y.mean():.4f}")

    res_df = pd.DataFrame(results).sort_values('Win_Pred_Acc', ascending=False)
    print("\n📊 Regression Evaluation Summary:")
    print(res_df)
    res_df.to_csv("meta_regression_summary_old.csv", index=False)
    return res_df

data = load_and_prepare_data()
if data is not None:
    results = train_method_regressors(data)

📂 Loading OLD version data (better coverage)...
✅ Loaded 1162 samples (62 datasets)
🧠 Meta-Features (70): ['Number_of_Instances', 'Number_of_Attributes', 'Dimensionality', 'Number_of_Missing_Values', 'Percentage_of_Missing_Values']...

🏋️ Training Regressors (Predicting Score Gain)...
  ✅ OneHotEncoder             | N= 11 | WinAcc=63.64% | AvgGain=-0.0012
  ✅ LabelEncoder              | N= 75 | WinAcc=60.00% | AvgGain=0.0010
  ✅ FrequencyEncoder          | N= 75 | WinAcc=66.67% | AvgGain=-0.0083
  ✅ StandardScaler            | N= 76 | WinAcc=57.89% | AvgGain=-0.0001
  ✅ FastICA                   | N=201 | WinAcc=91.04% | AvgGain=-0.0033
  ✅ OneHotEncoder_LowCard     | N= 64 | WinAcc=68.75% | AvgGain=-0.0002
  ✅ RareLabelEncoder          | N= 64 | WinAcc=57.81% | AvgGain=0.0016
  ✅ CategoricalInteractions   | N= 60 | WinAcc=86.67% | AvgGain=-0.0003
  ✅ ArithmeticInteractions    | N=178 | WinAcc=70.22% | AvgGain=0.0024
  ✅ ImputeMedianMode          | N= 22 | WinAcc=40.91% | AvgGain=0.001

## Inference

In [8]:
import pandas as pd
import numpy as np
import joblib
import os
from scipy.stats import skew, kurtosis, entropy

# ============================================================
# Configuration
# ============================================================
MODEL_DIR = "meta_models_regression"   # Regression models
CLASSIFICATION_MODEL = "meta_model_lgbm.pkl"
ENCODER_FILE = "meta_label_encoder.pkl"

# ============================================================
# Meta-Feature Extraction (70 features - matches training data)
# ============================================================

def get_quartiles(series):
    """Return Q1, Q2 (Median), Q3."""
    if len(series) == 0: 
        return 0, 0, 0
    return series.quantile([0.25, 0.5, 0.75]).tolist()

def calc_stats_full(series, prefix):
    """
    Calculate Min, Mean, Max, Std, Q1, Q2, Q3 for a series.
    This matches the FULL 70-feature extraction used in training.
    """
    if len(series) == 0:
        return {
            f'Min_{prefix}': 0, f'Mean_{prefix}': 0, f'Max_{prefix}': 0, f'Std_{prefix}': 0,
            f'Q1_{prefix}': 0, f'Q2_{prefix}': 0, f'Q3_{prefix}': 0
        }
    
    q1, q2, q3 = get_quartiles(series)
    return {
        f'Min_{prefix}': series.min(),
        f'Mean_{prefix}': series.mean(),
        f'Max_{prefix}': series.max(),
        f'Std_{prefix}': series.std(ddof=1) if len(series) > 1 else 0,
        f'Q1_{prefix}': q1,
        f'Q2_{prefix}': q2,
        f'Q3_{prefix}': q3
    }

def extract_meta_features_full(df, target_col=None, problem_type='classification'):
    """
    Extract 70 meta-features from a DataFrame.
    This matches the EXACT feature extraction used in training (importopenml.ipynb).
    
    Args:
        df: Input DataFrame
        target_col: Name of target column (will be excluded from feature analysis)
        problem_type: 'classification', 'binary', 'multiclass', or 'regression'
    
    Returns:
        DataFrame with one row of meta-features
    """
    features = {}
    
    # Separate X and y
    if target_col and target_col in df.columns:
        X = df.drop(columns=[target_col])
        y = df[target_col]
    else:
        X = df
        y = None
    
    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    
    n_instances = len(df)
    n_attrs = len(X.columns)
    
    # ========== Generic Features (49-61) ==========
    features['Number_of_Instances'] = n_instances
    features['Number_of_Attributes'] = n_attrs
    features['Dimensionality'] = n_attrs / n_instances if n_instances > 0 else 0
    
    missing_total = X.isnull().sum().sum()
    features['Number_of_Missing_Values'] = missing_total
    features['Percentage_of_Missing_Values'] = missing_total / (n_instances * n_attrs) if n_instances * n_attrs > 0 else 0
    
    rows_missing = X.isnull().any(axis=1).sum()
    features['Number_of_Instances_with_Missing_Values'] = rows_missing
    features['Percentage_of_Instances_with_Missing_Values'] = rows_missing / n_instances if n_instances > 0 else 0
    
    # Class-related features
    if y is not None and problem_type != 'regression':
        class_counts = y.value_counts()
        features['Number_of_Classes'] = len(class_counts)
        probs = class_counts / n_instances
        features['Class_Entropy'] = entropy(probs, base=2)
        features['Minority_Class_Size'] = class_counts.min()
        features['Majority_Class_Size'] = class_counts.max()
        features['Minority_Class_Percentage'] = class_counts.min() / n_instances
        features['Majority_Class_Percentage'] = class_counts.max() / n_instances
    else:
        features['Number_of_Classes'] = 0
        features['Class_Entropy'] = 0
        features['Minority_Class_Size'] = 0
        features['Majority_Class_Size'] = 0
        features['Minority_Class_Percentage'] = 0
        features['Majority_Class_Percentage'] = 0

    # ========== Continuous Attributes (1-26) ==========
    features['Number_of_Continuous_Attributes'] = len(num_cols)
    features['Percentage_of_Continuous_Attributes'] = len(num_cols) / n_attrs if n_attrs > 0 else 0
    
    if len(num_cols) > 0:
        X_num = X[num_cols]
        means = X_num.mean()
        stds = X_num.std()
        skews = X_num.apply(lambda x: skew(x.dropna()))
        kurtoses = X_num.apply(lambda x: kurtosis(x.dropna()))
        
        features.update(calc_stats_full(means, 'Means_of_Continuous_Attributes'))
        features.update(calc_stats_full(stds, 'Std_of_Continuous_Attributes'))
        features.update(calc_stats_full(skews, 'Skewness_of_Continuous_Attributes'))
        features.update(calc_stats_full(kurtoses, 'Kurtosis_of_Continuous_Attributes'))
    else:
        for stat in ['Means', 'Std', 'Skewness', 'Kurtosis']:
            features.update(calc_stats_full(pd.Series([]), f'{stat}_of_Continuous_Attributes'))

    # ========== Categorical & Binary Attributes (27-48) ==========
    binary_cols = [c for c in X.columns if X[c].nunique() == 2]
    features['Number_of_Categorical_Attributes'] = len(cat_cols)
    features['Number_of_Binary_Attributes'] = len(binary_cols)
    features['Percentage_of_Categorical_Attributes'] = len(cat_cols) / n_attrs if n_attrs > 0 else 0
    features['Percentage_of_Binary_Attributes'] = len(binary_cols) / n_attrs if n_attrs > 0 else 0
    
    if len(cat_cols) > 0:
        entropies = []
        mutual_infos = []  # Simplified: set to 0 for local inference
        distinct_values = []
        
        for c in cat_cols:
            col_data = X[c].astype(str)
            # Entropy
            vc = col_data.value_counts(normalize=True)
            entropies.append(entropy(vc, base=2))
            # Distinct Values
            distinct_values.append(col_data.nunique())
            # Mutual Information (simplified: set to 0 for speed)
            mutual_infos.append(0)

        features.update(calc_stats_full(pd.Series(entropies), 'Attribute_Entropy'))
        features.update(calc_stats_full(pd.Series(mutual_infos), 'Mutual_Information'))
        features.update(calc_stats_full(pd.Series(distinct_values), 'Attribute_Distinct_Values'))
        
        # Equivalent Number of Attributes & Noise to Signal Ratio
        features['Equivalent_Number_of_Attributes'] = 0
        features['Noise_to_Signal_Ratio'] = 0
        
    else:
        for stat in ['Attribute_Entropy', 'Mutual_Information', 'Attribute_Distinct_Values']:
            features.update(calc_stats_full(pd.Series([]), stat))
        features['Equivalent_Number_of_Attributes'] = 0
        features['Noise_to_Signal_Ratio'] = 0

    return pd.DataFrame([features])

# ============================================================
# Regression-based Recommendation (Recommended!)
# ============================================================

def recommend_preprocessing_regression(df, target_col=None, problem_type='classification'):
    """
    Use regression models to predict Score Gain for each preprocessing method.
    This is the RECOMMENDED approach.
    
    Returns:
        DataFrame with predicted gains for each method
    """
    print("🤖 Meta-Model Inference (Regression)...")
    
    if not os.path.exists(MODEL_DIR):
        print(f"❌ Model directory '{MODEL_DIR}' not found. Please train models first.")
        return None
    
    # 1. Extract Features
    print("   Extracting meta-features...")
    X_meta = extract_meta_features_full(df, target_col, problem_type)
    print(f"   ✅ Extracted {len(X_meta.columns)} features")
    
    # 2. Load and run each regressor
    predictions = []
    
    for model_file in os.listdir(MODEL_DIR):
        if not model_file.startswith('regressor_') or not model_file.endswith('.pkl'):
            continue
            
        method = model_file.replace('regressor_', '').replace('.pkl', '')
        
        try:
            model = joblib.load(os.path.join(MODEL_DIR, model_file))
            
            # Align features with what model expects
            feature_cols = model.feature_name_
            
            # Handle missing columns (fill with 0)
            for col in feature_cols:
                if col not in X_meta.columns:
                    X_meta[col] = 0
            
            X = X_meta[feature_cols]
            pred_gain = model.predict(X)[0]
            
            predictions.append({
                'Method': method,
                'Predicted_Gain': pred_gain,
                'Recommended': pred_gain > 0
            })
        except Exception as e:
            print(f"   ⚠️ Error loading {method}: {e}")
            continue
    
    if not predictions:
        print("❌ No predictions made. Check if models exist.")
        return None
    
    result = pd.DataFrame(predictions).sort_values('Predicted_Gain', ascending=False)
    
    print(f"\n📊 Preprocessing Recommendations:")
    print("-" * 50)
    for _, row in result.head(5).iterrows():
        status = "✅" if row['Recommended'] else "❌"
        print(f"   {status} {row['Method']:25s} | Gain: {row['Predicted_Gain']:+.4f}")
    print("-" * 50)
    
    recommended = result[result['Recommended']]
    if len(recommended) > 0:
        print(f"\n🏆 Top Recommended: {recommended.iloc[0]['Method']}")
    else:
        print("\n⚠️ No method predicted to improve performance. Use Baseline.")
    
    return result

# ============================================================
# Classification-based Recommendation (Legacy)
# ============================================================

def recommend_preprocessing_classification(df, target_col=None):
    """
    Use classification model to predict the "best" preprocessing method.
    NOTE: This approach has lower accuracy (~32%). Use regression instead.
    """
    print("🤖 Meta-Model Inference (Classification - Legacy)...")
    
    if not os.path.exists(CLASSIFICATION_MODEL) or not os.path.exists(ENCODER_FILE):
        print("❌ Model files not found. Please train the model first.")
        return None
        
    clf = joblib.load(CLASSIFICATION_MODEL)
    le = joblib.load(ENCODER_FILE)
    
    print("   Extracting meta-features...")
    X_meta = extract_meta_features_full(df, target_col)
    
    try:
        probs = clf.predict_proba(X_meta)[0]
        best_idx = np.argmax(probs)
        best_method = le.inverse_transform([best_idx])[0]
        confidence = probs[best_idx]
        
        print(f"   ✅ Prediction Complete.")
        print(f"   🏆 Recommended Method: {best_method} (Confidence: {confidence:.2%})")
        
        return best_method
        
    except Exception as e:
        print(f"❌ Prediction failed: {e}")
        return None

# ============================================================
# Main Function
# ============================================================

def suggest_preprocessing(df, target_col=None, method='regression'):
    """
    Main entry point for preprocessing recommendation.
    
    Args:
        df: Input DataFrame
        target_col: Name of target column
        method: 'regression' (recommended) or 'classification' (legacy)
    
    Returns:
        Recommended preprocessing method(s)
    """
    if method == 'regression':
        return recommend_preprocessing_regression(df, target_col)
    else:
        return recommend_preprocessing_classification(df, target_col)

# ============================================================
# Example Usage
# ============================================================
if __name__ == "__main__":
    # Create a dummy dataframe for testing
    df_test = pd.DataFrame({
        'A': np.random.rand(100),
        'B': np.random.rand(100) * 100,
        'C': np.random.choice(['a', 'b', 'c'], 100),
        'target': np.random.randint(0, 2, 100)
    })
    
    # Use regression-based recommendation (default)
    result = suggest_preprocessing(df_test, target_col='target', method='regression')


🤖 Meta-Model Inference (Regression)...
   Extracting meta-features...
   ✅ Extracted 70 features

📊 Preprocessing Recommendations:
--------------------------------------------------
   ✅ ArithmeticInteractions    | Gain: +0.0044
   ✅ LabelEncoder              | Gain: +0.0024
   ✅ RareLabelEncoder          | Gain: +0.0017
   ✅ ImputeMedianMode          | Gain: +0.0015
   ✅ FastICA                   | Gain: +0.0003
--------------------------------------------------

🏆 Top Recommended: ArithmeticInteractions


# Combine


In [10]:
# ============================================================
# COMBINE OLD AND NEW VERSIONS
# ============================================================
# This combines the best of both versions:
# - Old version: Better method coverage (18 methods/dataset)
# - New version: More datasets (110 vs 62)
# ============================================================

import pandas as pd
import os

# File paths
OLD_PERFORMANCE = "performance_labels.csv"
NEW_PERFORMANCE = "performance_labels_large.csv"
OLD_META = "meta_features_61.csv"
NEW_META = "meta_features_large.csv"

COMBINED_PERFORMANCE = "performance_labels_combined.csv"
COMBINED_META = "meta_features_combined.csv"

def combine_versions():
    """
    Combine old and new versions of data.
    
    Strategy:
    - Keep ALL records from old version (better coverage)
    - Add NEW datasets from new version (that don't exist in old)
    - Remove duplicates based on (dataset_id, method)
    """
    print("🔀 Combining Old and New Versions...")
    
    # ========== 1. Combine Performance Labels ==========
    print("\n📊 Combining Performance Labels...")
    
    df_old = pd.read_csv(OLD_PERFORMANCE)
    df_new = pd.read_csv(NEW_PERFORMANCE)
    
    print(f"  Old version: {len(df_old)} samples, {df_old['dataset_id'].nunique()} datasets")
    print(f"  New version: {len(df_new)} samples, {df_new['dataset_id'].nunique()} datasets")
    
    # Combine and remove duplicates (keep old version's record if duplicate)
    df_combined = pd.concat([df_old, df_new], ignore_index=True)
    df_combined = df_combined.drop_duplicates(subset=['dataset_id', 'method'], keep='first')
    
    # Save
    df_combined.to_csv(COMBINED_PERFORMANCE, index=False)
    print(f"  ✅ Combined: {len(df_combined)} samples, {df_combined['dataset_id'].nunique()} datasets")
    print(f"  Saved to: {COMBINED_PERFORMANCE}")
    
    # ========== 2. Combine Meta-Features ==========
    print("\n🧠 Combining Meta-Features...")
    
    df_meta_old = pd.read_csv(OLD_META)
    df_meta_new = pd.read_csv(NEW_META)
    
    print(f"  Old version: {len(df_meta_old)} datasets")
    print(f"  New version: {len(df_meta_new)} datasets")
    
    # Combine and remove duplicates (keep old version's record if duplicate)
    df_meta_combined = pd.concat([df_meta_old, df_meta_new], ignore_index=True)
    df_meta_combined = df_meta_combined.drop_duplicates(subset=['dataset_id'], keep='first')
    
    # Save
    df_meta_combined.to_csv(COMBINED_META, index=False)
    print(f"  ✅ Combined: {len(df_meta_combined)} datasets")
    print(f"  Saved to: {COMBINED_META}")
    
    # ========== 3. Summary ==========
    print("\n" + "="*60)
    print("📈 COMBINATION SUMMARY")
    print("="*60)
    
    # Calculate stats
    methods_per_dataset = df_combined.groupby('dataset_id')['method'].nunique()
    
    print(f"  Total datasets:        {df_combined['dataset_id'].nunique()}")
    print(f"  Total samples:         {len(df_combined)}")
    print(f"  Avg methods/dataset:   {methods_per_dataset.mean():.1f}")
    print(f"  Min methods/dataset:   {methods_per_dataset.min()}")
    print(f"  Max methods/dataset:   {methods_per_dataset.max()}")
    
    # Method distribution
    print("\n  Method Distribution:")
    method_counts = df_combined['method'].value_counts()
    for method, count in method_counts.head(10).items():
        print(f"    {method:25s}: {count}")
    
    return df_combined, df_meta_combined

# Run the combination
combined_labels, combined_meta = combine_versions()

🔀 Combining Old and New Versions...

📊 Combining Performance Labels...
  Old version: 1162 samples, 62 datasets
  New version: 886 samples, 110 datasets
  ✅ Combined: 899 samples, 110 datasets
  Saved to: performance_labels_combined.csv

🧠 Combining Meta-Features...
  Old version: 72 datasets
  New version: 110 datasets
  ✅ Combined: 110 datasets
  Saved to: meta_features_combined.csv

📈 COMBINATION SUMMARY
  Total datasets:        110
  Total samples:         899
  Avg methods/dataset:   8.2
  Min methods/dataset:   4
  Max methods/dataset:   16

  Method Distribution:
    Baseline                 : 110
    Binning_Quantile_10      : 101
    StandardScaler           : 101
    FastICA                  : 93
    ArithmeticInteractions   : 92
    PCA_95                   : 91
    SelectKBest_50pct        : 72
    RareLabelEncoder         : 38
    FrequencyEncoder         : 38
    LabelEncoder             : 38
